# 01 - EDA: Online Retail

Objetivo: entender patrones de compra, distribucion por pais y horas pico, y documentar que decisiones de modelado/feature engineering se derivan de cada hallazgo.

**Integrante 2 - ML & Feature Engineer**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/clean/online_retail_clean.csv", parse_dates=["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df.head()

## 1. Distribucion de ventas por pais

**Decision que depende de esto:** si un pais domina el dataset (ej. UK), evaluar si conviene modelar por separado o si el pais debe entrar como feature/segmento.

In [ ]:
country_sales = df.groupby("Country")["TotalPrice"].sum().sort_values(ascending=False)
country_sales.head(10).plot(kind="bar", figsize=(8,4), title="Ventas totales por pais (top 10)")
plt.show()

## 2. Horas y dias pico de compra

**Decision que depende de esto:** si hay estacionalidad horaria/semanal fuerte, considerar variables temporales en feature engineering (ej. % compras fin de semana, ya incluida en RFM ampliado).

In [ ]:
df["hour"] = df["InvoiceDate"].dt.hour
df["dayofweek"] = df["InvoiceDate"].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(12,4))
df.groupby("hour")["InvoiceNo"].nunique().plot(kind="bar", ax=axes[0], title="Transacciones por hora")
df.groupby("dayofweek")["InvoiceNo"].nunique().plot(kind="bar", ax=axes[1], title="Transacciones por dia")
plt.tight_layout()
plt.show()

## 3. Distribucion de gasto por cliente (Monetary)

**Decision que depende de esto:** si la distribucion esta muy sesgada (skewness alta), justifica el StandardScaler o incluso una transformacion log antes de clustering, y ayuda a fijar rangos razonables para las Data Quality Gates de Integrante 1.

In [ ]:
customer_spend = df.groupby("CustomerID")["TotalPrice"].sum()
print("Skewness:", customer_spend.skew())
sns.histplot(customer_spend, bins=50)
plt.title("Distribucion de gasto total por cliente")
plt.show()

## 4. Diversidad de productos y tasa de devoluciones

**Decision que depende de esto:** confirma si estas variables comportamentales (incluidas en `build_features.py`) aportan separacion entre clientes, mas alla de RFM puro.

In [ ]:
import sys
sys.path.append("..")
from src.features.build_features import build_customer_features

features, scaler, feature_cols = build_customer_features(df)
features[feature_cols].describe()

## Conclusiones y decisiones tomadas

### Comparacion de modelos probados

| Algoritmo        | Clusters encontrados | Silhouette Score | Davies-Bouldin |
|-------------------|----------------------|-------------------|-----------------|
| K-Means (k=3)      | 3                    | **0.8876**        | 0.5548          |
| K-Means (k=4)      | 4                    | 0.3654            | 0.7745          |
| K-Means (k=5)      | 5                    | 0.4316            | 0.6720          |
| DBSCAN (eps=0.5)   | 1 (todo ruido)       | No calculable     | No calculable   |
| DBSCAN (eps=0.8)   | 2                    | 0.5775            | **0.4148**      |
| Agglomerative (k=3)| 3                    | 0.8589            | 0.6116          |
| Agglomerative (k=4)| 4                    | 0.4069            | 0.7804          |
| Agglomerative (k=5)| 5                    | 0.4154            | 0.7309          |

### Modelo elegido: K-Means con k=3

Se selecciono K-Means (k=3) como modelo de Produccion porque obtuvo el Silhouette Score mas alto de las 8 configuraciones probadas (0.8876, en una escala de -1 a 1 donde valores cercanos a 1 indican clusters bien separados y compactos). Agglomerative Clustering con k=3 obtuvo un resultado muy cercano (0.8589), lo que refuerza la conclusion: **la estructura natural de los clientes de este dataset se explica mejor con 3 grupos, no con 4 o 5**. Al aumentar a k=4 o k=5 en ambos algoritmos, el Silhouette cae drasticamente (de ~0.88 a ~0.40), lo que indica que forzar mas grupos de los que existen naturalmente en los datos rompe la cohesion interna de los clusters.

DBSCAN se comporto distinto al resto: con eps=0.5 no encontro estructura (clasifico todo como ruido), y con eps=0.8 solo encontro 2 clusters. Aunque su Davies-Bouldin (0.4148) fue el mejor de la tabla, su Silhouette (0.5775) fue notablemente menor que el de K-Means/Agglomerative con k=3. Esto sugiere que los clientes no forman regiones de densidad claramente separadas (como asume DBSCAN), sino grupos mas compactos y esfericos, que es justamente lo que K-Means asume por diseño — coherente con el resultado obtenido.

### Que decision de modelado se deriva de esto

- Se descarta DBSCAN para este dataset: la naturaleza de las variables RFM (con clientes concentrados en rangos similares de recencia/frecuencia y una cola larga de gasto) no genera las zonas de densidad distintas que DBSCAN necesita para funcionar bien.
- Se confirma k=3 como el numero de segmentos de clientes a usar en produccion, no k=4 ni k=5 — mas grupos no mejoran la calidad del clustering, solo lo fragmentan.
- El criterio de seleccion fue el Silhouette Score como metrica principal (mide que tan bien separados y compactos quedan los clusters), usando Davies-Bouldin como metrica secundaria de referencia, no como criterio de desempate contra el Silhouette.
